# Chapter 10 — Recursively Recurse with Recursion

## What recursion is

A function that calls **itself**.\
Used correctly, it solves certain problems — especially ones with an unknown "depth" — in surprisingly simple ways.\
Used incorrectly (no stopping condition), it causes **infinite recursion**, which crashes the program.

This chapter connects directly back to last chapter's stacks — Python 
itself uses a stack internally to make recursion work, which is the real 
payoff of understanding stacks first.

## Recursion vs. a loop — same result, different mechanism

Anything you can do with a loop, you can also do with recursion.\
This doesn't mean recursion is always better — for simple repetition, a loop is often just as good or clearer.\
Recursion becomes genuinely valuable later, for a specific category of problem (unknown depth — see filesystem example below).

In [ ]:
def countdown(number):
    print(number)
    countdown(number - 1)     # calls itself again — but NEVER STOPS

# countdown(10)   # DON'T RUN — this recurses forever into negative numbers,
                     # eventually crashing with a RecursionError

## The base case — the essential stopping condition

Every recursive function needs at least one **base case**:\
a condition under which the function returns *without* calling itself again.\
Without one, recursion never terminates.

In [ ]:
def countdown(number):
    print(number)
    if number == 0:
        return          # BASE CASE — stops the recursion
    else:
        countdown(number - 1)

countdown(10)

## How to *read* recursive code — the "napkin method"

Recursive code is confusing to trace top-down. The book's recommended 
technique instead works **bottom-up**, from the base case outward:

1. Identify the base case.
2. Work out what happens *at* the base case.
3. Work out the case just *before* the base case, using the base case's 
   already-known result.
4. Repeat, building up one case at a time.

This is A SKILL TO PRACTICE — trying to mentally 
trace a recursive call top-down (as you would a loop) 

In [ ]:
def factorial(number):
    if number == 1:
        return 1                          # base case
    else:
        return number * factorial(number - 1)

# Napkin, built bottom-up:
# factorial(1) = 1                          <- base case, no recursion
# factorial(2) = 2 * factorial(1) = 2 * 1 = 2
# factorial(3) = 3 * factorial(2) = 3 * 2 = 6
# factorial(4) = 4 * factorial(3) = 4 * 6 = 24

factorial(5)   # 120

## How the computer actually executes recursion — the call stack

Here's the tricky part: when `factorial(3)` calls `factorial(2)`, 
`factorial(3)` **isn't finished yet** — it's paused, waiting on the result 
of `factorial(2)`. And `factorial(2)` is itself paused waiting on 
`factorial(1)`. The computer needs a way to remember "what am I still in 
the middle of, and in what order do I need to finish them?"

**This is solved with an actual stack** (the same LIFO structure from Ch. 9) 
called the **call stack**:

- Each time a function calls another (including calling itself), the 
  *paused, unfinished* call gets **pushed** onto the call stack.
- When the innermost call (the base case) finally completes, the computer 
  **pops** the call stack to find out which paused function to resume next.
- Since it's a stack, the *most recently paused* function is always the 
  *next one resumed* — LIFO — which is exactly correct, since the most 
  recent call is the one most immediately blocking everything above it.

**Trace for `factorial(3)`:**

Call order (pushed): factorial(3) → factorial(2) → factorial(1)\
Completion order (popped): factorial(1) → factorial(2) → factorial(3)


Each function passes its result **up** to the function that called it — 
`factorial(1)` returns `1` to `factorial(2)`, which computes `2 * 1 = 2` and 
returns that to `factorial(3)`, which computes `3 * 2 = 6`.

## Stack overflow — infinite recursion's real failure mode

Without a base case (like the broken `countdown` example above), the call stack keeps growing forever — one new paused call pushed on every recursive call, nothing ever popped.\
Eventually, the computer runs out of memory allocated for the call stack, and crashes with a **stack overflow** error.\
In Python, this shows up as `RecursionError: maximum recursion depth exceeded`.

In [ ]:
import sys
print(sys.getrecursionlimit())   # Python's default cap, usually 1000

def broken(n):
    return broken(n + 1)         # no base case at all

# broken(0)   # uncomment to see RecursionError in action

## Where recursion genuinely shines — unknown depth

Loops require you to know (or compute) how many times to repeat in advance, 
or at least structure nested loops for a **fixed** number of levels. Some 
problems have **arbitrary, unknown depth** — you don't know how many levels 
deep the structure goes until you're actually traversing it.

**Classic example: traversing a filesystem.** A directory can contain 
subdirectories, which contain more subdirectories, arbitrarily deep. A loop 
would need one nested loop *per level* — impossible to write in advance if 
the depth is unknown. Recursion handles this naturally: the function calls 
itself on each subdirectory it finds, automatically going as deep as needed.

In [ ]:
import os

def find_directories(directory):
    for filename in os.listdir(directory):
        full_path = os.path.join(directory, filename)
        if os.path.isdir(full_path):
            print(full_path)
            find_directories(full_path)      # recurse into the subdirectory

# find_directories(".")

**The pattern to remember**: whenever a problem involves "go arbitrarily 
deep into a nested structure" — filesystems, nested lists/dicts, tree 
structures, JSON with unknown nesting — recursion is very often the 
natural fit, precisely because it doesn't need to know the depth in advance.

## Exercises (solutions on p. 447)

1. **Identify the base case** in this function, which prints every other 
   number from `low` to `high`:

In [ ]:
def print_every_other(low, high):
    if low > high:
        return
    print(low)
    print_every_other(low + 2, high)

2. **Predict the outcome**: what happens when `factorial(10)` runs, if the 
   recursive call is `factorial(n - 2)` instead of `factorial(n - 1)`, but 
   the base case still only checks `n == 1`?

In [ ]:
def factorial(n):
    if n == 1:
        return 1
    return n * factorial(n - 2)

3. **Fix the missing base case** — this function is meant to sum all 
   numbers from `low` to `high`, but has no base case and will recurse forever:

In [ ]:
def total(low, high):
    return high + total(low, high - 1)

4. **Write a recursive function** that prints every number (ignoring nested 
   arrays) from a deeply nested list:

In [ ]:
data = [1, 2, 3, [4, 5, 6], 7, [8, [9, 10, 11, [12, 13, 14]]],
        [15, 16, 17, 18, 19, [20, 21, 22, [23, 24, 25, [26, 27, 29]], 30, 31], 32], 33]